In [ ]:
# -*- coding: utf-8 -*-
"""
Eccentric Gear Fault Diagnosis — Data Leakage Study
Condition: T3, S600
Model: C-BiLSTM (1D-CNN Feature Extractor + BiLSTM)
Task: Evaluate model performance and accuracy gap across different temporal safety gaps.
"""

import os
import sys
import random
import logging
import warnings
import numpy as np
import pandas as pd
import scipy.io as sio

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score

warnings.filterwarnings('ignore')

# ==========================================================================
# 1. CONFIGURATION
# ==========================================================================
_SAMPLING_RATE = 51200

class CFG:
    # =========================================================
    # [USER CONFIGURABLE] Dataset Path and Files
    # =========================================================
    FOLDER_PATH = r'F:\GearEccDataset\G1\T3\S600'
    
    FILE_NAMES = [
        'N9_G1_E00_T3_S600.mat', 'N25_G1_E02_T3_S600.mat', 'N41_G1_E04_T3_S600.mat',
        'N57_G1_E06_T3_S600.mat', 'N73_G1_E08_T3_S600.mat', 'N89_G1_E10_T3_S600.mat',
        'N105_G1_E12_T3_S600.mat', 'N121_G1_E14_T3_S600.mat', 'N137_G1_E16_T3_S600.mat',
        'N153_G1_E18_T3_S600.mat'
    ]
    
    TARGET_KEY  = 'DataSensor03'
    
    NUM_CLASSES = 10
    SAMPLING_RATE = _SAMPLING_RATE
    TOTAL_POINTS  = 119808
    SAMPLE_LENGTH = 2048
    
    OVERLAP_RATES = [0.75]
    SPLIT_RATIO = (0.7, 0.2, 0.1)
    
    # Specific temporal gaps to evaluate data leakage
    TIME_GAPS_SEC = [0.02, 0.04, 0.05, 0.06, 0.0667, 0.08, 0.10]
    MECHANISM_GAPS = [int(t * _SAMPLING_RATE) for t in TIME_GAPS_SEC]

    SEEDS       = [42, 123, 456, 789, 1010]
    BATCH_SIZE  = 64
    EPOCHS      = 100  
    LR          = 1e-3
    WEIGHT_DECAY= 1e-4
    DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    RESULT_DIR  = './C_BiLSTM_Results'
    TABLE_DIR   = './C_BiLSTM_Results/Data_Tables'
    LOG_FILE    = './C_BiLSTM_Results/experiment_table_S600.log'

for d in (CFG.RESULT_DIR, CFG.TABLE_DIR): 
    os.makedirs(d, exist_ok=True)

# ==========================================================================
# 2. UTILS & DATA LOADER
# ==========================================================================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

def get_logger():
    logger = logging.getLogger('GearEcc_CBiLSTM_S600')
    logger.setLevel(logging.INFO)
    if not logger.handlers:
        fmt = logging.Formatter('[%(asctime)s] %(message)s', datefmt='%H:%M:%S')
        sh = logging.StreamHandler(sys.stdout)
        sh.setFormatter(fmt)
        fh = logging.FileHandler(CFG.LOG_FILE, mode='w', encoding='utf-8')
        fh.setFormatter(fmt)
        logger.addHandler(sh)
        logger.addHandler(fh)
    return logger

LOGGER = get_logger()

def _load_mat_any_version(path, target_key):
    try:
        mat = sio.loadmat(path)
        for k in mat.keys():
            if k.lower() == target_key.lower(): 
                return np.asarray(mat[k]).squeeze()
        raise KeyError
    except:
        import h5py
        with h5py.File(path, 'r') as f:
            keys = list(f.keys())
            match = next((k for k in keys if k.lower() == target_key.lower()), None)
            return np.array(f[match]).squeeze()

def load_raw_signals():
    raw = {}
    for label, fname in enumerate(CFG.FILE_NAMES):
        path = os.path.join(CFG.FOLDER_PATH, fname)
        sig = _load_mat_any_version(path, CFG.TARGET_KEY).astype(np.float32).ravel()
        raw[label] = sig[:CFG.TOTAL_POINTS]
    return raw

def slice_signal(signal, win, overlap):
    step = max(1, int(round(win * (1 - overlap))))
    starts = np.arange(0, len(signal) - win + 1, step)
    samples = np.stack([signal[s:s+win] for s in starts], axis=0)
    return samples, starts

def zscore(x):
    mu, sd = x.mean(axis=1, keepdims=True), x.std(axis=1, keepdims=True) + 1e-8
    return (x - mu) / sd

def build_dataset(raw, overlap):
    X, y, starts = [], [], []
    for label, sig in raw.items():
        samp, st = slice_signal(sig, CFG.SAMPLE_LENGTH, overlap)
        X.append(samp)
        y.append(np.full(len(samp), label))
        starts.append(st)
    return zscore(np.concatenate(X)), np.concatenate(y), np.concatenate(starts)

class SignalDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float().unsqueeze(1)
        self.y = torch.from_numpy(y).long()
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

def split_indices(X, y, starts, seed, ratios, gap):
    rng = np.random.RandomState(seed)
    train_r, val_r, test_r = ratios
    idx_tr, idx_va, idx_te = [], [], []
    for c in np.unique(y):
        cls_idx = np.where(y == c)[0]
        cls_idx = cls_idx[np.argsort(starts[cls_idx])]
        n = len(cls_idx)
        n_tr = int(n * train_r)
        n_va = int(n * val_r)
        tr = cls_idx[:n_tr]
        va = cls_idx[n_tr:n_tr+n_va]
        te = cls_idx[n_tr+n_va:]

        if len(tr) and len(te):
            tr_end_t = starts[tr].max() + CFG.SAMPLE_LENGTH
            va = va[starts[va] >= tr_end_t + gap] if len(va) else va
            va_end_t = starts[va].max() + CFG.SAMPLE_LENGTH if len(va) else tr_end_t
            te = te[starts[te] >= va_end_t + gap]

        idx_tr.append(tr)
        idx_va.append(va)
        idx_te.append(te)
    return np.concatenate(idx_tr), np.concatenate(idx_va), np.concatenate(idx_te)

# ==========================================================================
# 3. MODEL (C-BiLSTM)
# ==========================================================================
class BiLSTM(nn.Module):
    def __init__(self, num_classes=10):
        super(BiLSTM, self).__init__()
        
        self.feature_extractor = nn.Sequential(
            nn.Conv1d(in_channels=1, out_channels=16, kernel_size=64, stride=8, padding=28),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2)
        )
        
        self.hidden_size = 64
        self.num_layers = 2
        
        self.lstm = nn.LSTM(
            input_size=16, 
            hidden_size=self.hidden_size, 
            num_layers=self.num_layers, 
            batch_first=True, 
            bidirectional=True
        )
        
        self.fc1 = nn.Linear(self.hidden_size * 2, 64)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.feature_extractor(x)
        x = x.permute(0, 2, 1)
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        out = self.fc1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        return out

# ==========================================================================
# 4. TRAIN & EVALUATE
# ==========================================================================
def evaluate_loader(model, loader):
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for x, yt in loader:
            ps.append(model(x.to(CFG.DEVICE)).argmax(1).cpu().numpy())
            ys.append(yt.numpy())
    return accuracy_score(np.concatenate(ys), np.concatenate(ps))

def train_one_run(X, y, idx_tr, idx_va, idx_te, seed):
    set_seed(seed)
    tr_loader = DataLoader(SignalDataset(X[idx_tr], y[idx_tr]), batch_size=CFG.BATCH_SIZE, shuffle=True)
    va_loader = DataLoader(SignalDataset(X[idx_va], y[idx_va]), batch_size=CFG.BATCH_SIZE, shuffle=False)
    te_loader = DataLoader(SignalDataset(X[idx_te], y[idx_te]), batch_size=CFG.BATCH_SIZE, shuffle=False)

    model = BiLSTM(CFG.NUM_CLASSES).to(CFG.DEVICE)
    opt = optim.Adam(model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)
    crit = nn.CrossEntropyLoss()

    best_va, best_state = -1, None
    for ep in range(CFG.EPOCHS):
        model.train()
        for x, yt in tr_loader:
            opt.zero_grad()
            crit(model(x.to(CFG.DEVICE)), yt.to(CFG.DEVICE)).backward()
            opt.step()
        
        va_acc = evaluate_loader(model, va_loader)
        if va_acc > best_va:
            best_va = va_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    
    train_acc = evaluate_loader(model, tr_loader)
    test_acc = evaluate_loader(model, te_loader)
    
    return train_acc, test_acc

# ==========================================================================
# 5. MAIN EXPERIMENT LOOP & TABLE GENERATION
# ==========================================================================
def run_experiments():
    LOGGER.info('-'*70)
    LOGGER.info(f'Starting C-BiLSTM Experiments | Condition: T3 S600')
    LOGGER.info(f'Overlap: {CFG.OVERLAP_RATES[0]} | Epochs: {CFG.EPOCHS} | Seeds: {CFG.SEEDS}')
    LOGGER.info('-'*70)

    raw_results = []

    try: 
        raw = load_raw_signals()
    except Exception as e: 
        LOGGER.error(f"Failed to load data: {e}")
        return
        
    X, y, starts = build_dataset(raw, overlap=CFG.OVERLAP_RATES[0])
    
    for gap, time_sec in zip(CFG.MECHANISM_GAPS, CFG.TIME_GAPS_SEC):
        test_accs = []
        acc_gaps = []
        
        for seed in CFG.SEEDS:
            try:
                idx_tr, idx_va, idx_te = split_indices(X, y, starts, seed, CFG.SPLIT_RATIO, gap)
                if len(idx_tr) > 0 and len(idx_te) > 0:
                    train_acc, test_acc = train_one_run(X, y, idx_tr, idx_va, idx_te, seed)
                    test_accs.append(test_acc)
                    acc_gaps.append(train_acc - test_acc)
            except Exception as e:
                LOGGER.warning(f"Seed {seed} failed for gap {time_sec}s: {e}")
        
        if test_accs:
            test_mean, test_std = np.mean(test_accs), np.std(test_accs, ddof=1)
            gap_mean, gap_std = np.mean(acc_gaps), np.std(acc_gaps, ddof=1)
            
            acc_str = f"{test_mean:.2f} ± {test_std:.2f}"
            gap_str = f"{gap_mean:.2f} ± {gap_std:.2f}"
            
            LOGGER.info(f'  Gap {time_sec:6.4f}s | Acc: {acc_str} | Acc Gap: {gap_str}')
        else:
            acc_str, gap_str = "N/A", "N/A"

        raw_results.append({
            'Gap_s': time_sec,
            'Acc': acc_str,
            'Acc Gap': gap_str
        })

    # ==========================================================================
    # Generate Transposed Table for Single Condition
    # ==========================================================================
    df = pd.DataFrame(raw_results)
    df.set_index('Gap_s', inplace=True)
    
    # Transpose so Metrics are rows and Gaps are columns
    df_transposed = df.T
    df_transposed.columns = [f"{col}s" for col in df_transposed.columns]
    df_transposed.index.name = 'Metric'

    # ==========================================================================
    # Print and Export
    # ==========================================================================
    LOGGER.info("\n" + "="*80)
    LOGGER.info("FINAL RESULTS TABLE (T3 S600 | Mean ± Std)")
    LOGGER.info("="*80)
    LOGGER.info("\n" + df_transposed.to_string())
    LOGGER.info("="*80)
    
    excel_path = os.path.join(CFG.TABLE_DIR, 'Final_Results_Table_CBiLSTM_S600.xlsx')
    df_transposed.to_excel(excel_path)
    LOGGER.info(f"\nTable successfully exported to: {excel_path}")

if __name__ == '__main__':
    run_experiments()
